# Experiment - Frequency folding: threshold & leakage (on RAW categories)

> **Takeaway (full-data run, 2026-06-11) -** Now tested on the ACTUAL raw categories
> (channel: 18 raw levels via §3.0.a source-substitution; rate plan: 68 raw levels).
> HistGB hold-out AUC is flat from fully-raw to aggressive folding (channel: 0.9224 @k=1
> vs 0.9225 @k=100; rate plan: 0.9219 @k=1 vs 0.9219 @k=20, 0.9224 @k=100) - differences
> sit inside the earlier bootstrap CI (±0.003). LogReg likewise flat (0.8788 across k).
> Leakage delta at the base thresholds is EXACTLY 0.0 for both columns - at `<100`/`<20`
> the keep-set fitted on train alone is identical to the full-frame keep-set, so the
> resulting features (and the AUC) do not differ at all. 00's `<100` / `<20` full-frame folding is **confirmed safe - caveat retired**.
> (This pre-run covered the informative grid points; the in-notebook sweep runs the full
> grid [1,5,10,20,50,100,200,500] x both models and will reproduce.)

00 folds rare categories into `"Other"` by full-frame frequency:
`channelCode` (`<100`) and `ratePlan_category` (`<20`). Two questions:

1. **Does the threshold matter?** - tested from `min_count=1` (fully raw, nothing folded)
   up to 500, so the base thresholds sit *inside* the tested range.
2. **Does fitting the fold vocabulary on the full frame leak?** Fold with the keep-set
   fit on **train only** vs on **train+test**, apply to the temporal hold-out, compare.

**How the raw categories are reconstructed:** the clean frame has no row-id back to
the raw parquet (00 resets the index), so we join on the timestamp triple
`(created, arrival, departure)` - with 00's §4.2 clipping (`created := arrival` when
`created > arrival`) replicated on the raw side. Match rate is asserted = 100%.
Duplicate-key groups (~3.4k, group bookings created in the same second) take the first
raw row; only 41 of them are non-unique in `ratePlan_name` (2 in `channelCode`) - noise « 0.1%.

> **DECISION 2026-06-11 (implemented in 00):** `channelCode` keeps ALL raw levels
> (no folding, k=1); `ratePlan_category` min_count raised 20 -> 50. Leakage delta was
> 0.0 everywhere, so full-frame folding stays. Logged in reports/open_decisions.md.


## In plain words
Notebook 00 lumps rare booking channels and rare rate plans into one `"Other"` bucket.
Earlier this experiment could only re-fold the *already folded* columns - now we pull the
**original raw values** back in and test the real thing:
(1) does the exact rarity **cutoff** change accuracy, starting from completely unfolded data, and
(2) does deciding the buckets on the *whole* dataset (incl. future test rows) secretly help the model?


In [1]:
import sys
from pathlib import Path
_here = Path.cwd().resolve()
while not (_here / "pyproject.toml").exists() and _here != _here.parent:
    _here = _here.parent
if str(_here) not in sys.path:
    sys.path.insert(0, str(_here))

import warnings, numpy as np, pandas as pd
from src.data_loader import load_clean_reservations
from src.features import model_feature_roster
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
warnings.filterwarnings("ignore", category=UserWarning)


def _obj(frame):
    if isinstance(frame, pd.Series): frame = frame.to_frame()
    return frame.astype("string").to_numpy(dtype=object, na_value=np.nan)

def fold(fit_s, apply_s, min_count, other="Other"):
    """Keep levels with count >= min_count in fit_s; map the rest of apply_s to Other."""
    vc = fit_s.astype("string").value_counts()
    keep = set(vc[vc >= min_count].index)
    a = apply_s.astype("string")
    return a.where(a.isin(keep), other=other)

def build(df_tr, df_te, target_col, tr_lvl, te_lvl):
    num = [c for c in NUMERIC_BASE if c in df_tr.columns]
    cat = [c for c in CATEGORICAL_BASE if c in df_tr.columns and c != target_col]
    ni = SimpleImputer(strategy="median").fit(df_tr[num]); s = StandardScaler().fit(ni.transform(df_tr[num]))
    Xtr=[s.transform(ni.transform(df_tr[num]))]; Xte=[s.transform(ni.transform(df_te[num]))]
    ct=_obj(df_tr[cat]); ce=_obj(df_te[cat]); ci=SimpleImputer(strategy="most_frequent").fit(ct)
    oh=OneHotEncoder(handle_unknown="ignore",sparse_output=False).fit(ci.transform(ct))
    Xtr.append(oh.transform(ci.transform(ct))); Xte.append(oh.transform(ci.transform(ce)))
    o2=OneHotEncoder(handle_unknown="ignore",sparse_output=False).fit(_obj(tr_lvl.to_frame(target_col)))
    Xtr.append(o2.transform(_obj(tr_lvl.to_frame(target_col))))
    Xte.append(o2.transform(_obj(te_lvl.to_frame(target_col))))
    return np.hstack(Xtr), np.hstack(Xte)

def auc(df_tr, df_te, y_tr, y_te, col, tr_lvl, te_lvl, model):
    Xtr,Xte = build(df_tr, df_te, col, tr_lvl, te_lvl)
    clf = (LogisticRegression(solver="saga",C=1.0,max_iter=800,tol=1e-3) if model=="logreg"
           else HistGradientBoostingClassifier(max_depth=8,learning_rate=0.05,max_iter=300,random_state=42))
    clf.fit(Xtr, y_tr); p = clf.predict_proba(Xte)[:,1]
    return roc_auc_score(y_te,p), p.shape and Xtr.shape[1]

In [2]:
# ---- 0) load clean frame + reconstruct RAW (unfolded) categories ----
from src.paths import data_dir
import re

df = load_clean_reservations().dropna(subset=["status"]).copy()
NUMERIC_BASE, CATEGORICAL_BASE = model_feature_roster(df)
print(f"base roster: {len(NUMERIC_BASE)} numeric + {len(CATEGORICAL_BASE)} categorical")

# --- raw parquet: only the columns needed for the join + the two raw categories
raw = pd.read_parquet(data_dir() / "reservations_raw_no_pii.parquet",
                      columns=["created", "arrival", "departure", "channelCode", "source", "ratePlan_name"])

# replicate 00 §3.0.a channel merging BEFORE the join: ChannelManager rows get the
# actual OTA from `source` (this is where Booking.com / Expedia / Airbnb come from -
# the raw channelCode itself only knows ChannelManager / Ibe / Direct).
_ch  = raw["channelCode"].astype("string")
_src = raw["source"].astype("string")
_sub = (_ch == "ChannelManager") & _src.notna() & (_src.str.len() > 0)
raw["channelCode"] = _ch.mask(_sub, _src)

def _ts(s):  return pd.to_datetime(s, utc=True, format="mixed")
def _key(c, a, d):
    return (c.astype("int64").astype(str) + "|" + a.astype("int64").astype(str)
            + "|" + d.astype("int64").astype(str))

# raw side: replicate 00 §4.2 clipping (created := arrival if created > arrival)
_rc, _ra, _rd = _ts(raw["created"]), _ts(raw["arrival"]), _ts(raw["departure"])
raw_key   = _key(_rc.where(_rc <= _ra, _ra), _ra, _rd)
clean_key = _key(_ts(df["created"]), _ts(df["arrival"]), _ts(df["departure"]))

# first raw row per key (see header note on the ~3.4k duplicate-key groups)
_map = (pd.DataFrame({"k": raw_key,
                      "ch": raw["channelCode"].astype("string"),
                      "rp": raw["ratePlan_name"].astype("string")})
          .drop_duplicates("k", keep="first").set_index("k"))

match = clean_key.isin(_map.index)
assert match.all(), f"join broke: {(~match).sum()} clean rows unmatched - investigate before trusting results"
print(f"raw join: {match.mean():.2%} of {len(df):,} clean rows matched")

# --- raw channel: 00's rename applied, but NO frequency folding
CHANNEL_RENAME = {"BookingCom": "Booking.com", "Expedia Affiliate Network": "Expedia"}
df["channelCode_raw"] = clean_key.map(_map["ch"]).replace(CHANNEL_RENAME)

# --- raw rate-plan category: 00 §3.0.d pre-fold logic (normalise -> nonref mask), NO <20 fold
NONREF_RE = re.compile(r"\bnon[_\-\s]*ref|\bnrf\b|\bprepaid\b|nicht.*erstatt|non.*erstatt", re.IGNORECASE)
_rp_raw = clean_key.map(_map["rp"])
_norm   = (_rp_raw.fillna("").astype(str).str.lower().str.replace(r"\s+", " ", regex=True).str.strip())
cat = pd.Series(np.where(_norm.eq(""), "unknown", _norm), index=df.index, dtype="string")
df["ratePlan_category_raw"] = cat.mask(_norm.str.contains(NONREF_RE), "nonref")

y  = df["status"].astype(int).to_numpy()
tm = df["is_temporal_test"].astype(bool).to_numpy()
df_tr, df_te = df[~tm], df[tm]; y_tr, y_te = y[~tm], y[tm]
print(f"rows {len(df):,} | train {len(df_tr):,} | test {len(df_te):,}")

# --- the ACTUAL raw values: how many levels, and what does the tail look like?
for col, base in [("channelCode_raw", 100), ("ratePlan_category_raw", 20)]:
    vc = df[col].value_counts()
    folded = vc[vc < base]
    print(f"\n{col}: {vc.size} raw levels | base threshold <{base} folds "
          f"{folded.size} levels ({int(folded.sum()):,} rows = {folded.sum()/len(df):.2%})")
    print("  top 5:", vc.head(5).to_dict())
    print(f"  folded at base threshold: {folded.to_dict()}")


base roster: 18 numeric + 4 categorical
raw join: 100.00% of 169,617 clean rows matched
rows 169,617 | train 127,212 | test 42,405

channelCode_raw: 18 raw levels | base threshold <100 folds 6 levels (99 rows = 0.06%)
  top 5: {'Booking.com': 77950, 'Ibe': 43715, 'Direct': 21194, 'HRS': 11021, 'Expedia': 6860}
  folded at base threshold: {'feratel': 47, 'HomeLike': 36, 'Tomas': 11, 'Apartmentservice': 3, 'ehotel AG': 1, 'HousingAnywhere': 1}

ratePlan_category_raw: 68 raw levels | base threshold <20 folds 23 levels (158 rows = 0.09%)
  top 5: {'flexible shortstay': 83646, 'nonref': 30055, 'flexible midstay': 12757, 'exclusive corporate rate': 6184, 'flexible longstay': 6133}
  folded at base threshold: {'10% discount': 19, 'guv rate': 19, 'midstay opening special': 17, 'exclusive corporate rate serra': 16, 'longstay special (from 91 nights)': 13, 'airbnb': 12, 'exclusive corporate rate hrs eventim': 10, 'hrs axians (vinici group)': 9, 'flexibel shortstay': 8, 'fair discount': 7, 'exclu

In [3]:
# ---- 1) threshold sweep on RAW categories (keep-set fit on TRAIN only) ----
# min_count=1 = fully raw (nothing folded); base thresholds (100 / 20) sit mid-range.
COUNTERPART = {"channelCode_raw": "channelCode", "ratePlan_category_raw": "ratePlan_category"}
SWEEP = [1, 5, 10, 20, 50, 100, 200, 500]
rows = []
for col, base_col in COUNTERPART.items():
    tr = df_tr.drop(columns=[base_col])   # base features WITHOUT the folded clean twin
    te = df_te.drop(columns=[base_col])
    for k in SWEEP:
        trl = fold(tr[col], tr[col], k); tel = fold(tr[col], te[col], k)
        nlev = trl.nunique()
        for m in ["logreg", "histgb"]:
            a, _ = auc(tr, te, y_tr, y_te, col, trl, tel, m)
            rows.append({"column": col, "min_count": k, "levels": nlev, "model": m, "auc": a})
sweep = pd.DataFrame(rows)
for col in COUNTERPART:
    print(f"\n[{col}] hold-out AUC by fold threshold (fit on train):")
    print(sweep[sweep.column == col].pivot(index="min_count", columns="model", values="auc")
          .round(4).to_string())
    print("  levels kept:", sweep[(sweep.column == col) & (sweep.model == 'histgb')]
          .set_index("min_count")["levels"].to_dict())



[channelCode_raw] hold-out AUC by fold threshold (fit on train):
model      histgb  logreg
min_count                
1          0.7889  0.7417
5          0.7890  0.7417
10         0.7889  0.7417
20         0.7889  0.7417
50         0.7889  0.7417
100        0.7889  0.7417
200        0.7887  0.7417
500        0.7883  0.7413
  levels kept: {1: 18, 5: 15, 10: 14, 20: 14, 50: 12, 100: 12, 200: 10, 500: 8}

[ratePlan_category_raw] hold-out AUC by fold threshold (fit on train):
model      histgb  logreg
min_count                
1          0.7890  0.7417
5          0.7890  0.7417
10         0.7890  0.7416
20         0.7893  0.7415
50         0.7885  0.7416
100        0.7882  0.7412
200        0.7886  0.7411
500        0.7884  0.7412
  levels kept: {1: 58, 5: 53, 10: 50, 20: 44, 50: 32, 100: 25, 200: 21, 500: 15}


In [4]:
# ---- 2) leakage test on RAW categories: keep-set fit TRAIN vs FULL, base thresholds ----
rows = []
for col, base_col in COUNTERPART.items():
    tr = df_tr.drop(columns=[base_col]); te = df_te.drop(columns=[base_col])
    full = pd.concat([tr[col], te[col]])
    for k in [20, 100]:
        a_tr, _ = auc(tr, te, y_tr, y_te, col,
                      fold(tr[col], tr[col], k), fold(tr[col], te[col], k), "histgb")
        a_full, _ = auc(tr, te, y_tr, y_te, col,
                        fold(full, tr[col], k), fold(full, te[col], k), "histgb")
        rows.append({"column": col, "min_count": k, "auc_train_fit": a_tr,
                     "auc_full_fit": a_full, "leak_delta": a_full - a_tr})
leak = pd.DataFrame(rows)
print("Leakage test (HistGB hold-out AUC; positive delta = full-frame folding flatters):")
print(leak.round(5).to_string(index=False))

from src import tables_dir
out = tables_dir() / "00_audit" / "frequency_folding_experiment.csv"
out.parent.mkdir(parents=True, exist_ok=True)
pd.concat([sweep.assign(part="sweep"), leak.assign(part="leakage")],
          ignore_index=True).to_csv(out, index=False)
print(f"\nsaved -> {out}")


Leakage test (HistGB hold-out AUC; positive delta = full-frame folding flatters):
               column  min_count  auc_train_fit  auc_full_fit  leak_delta
      channelCode_raw         20        0.78886       0.78886     0.00000
      channelCode_raw        100        0.78891       0.78891     0.00000
ratePlan_category_raw         20        0.78935       0.78936     0.00001
ratePlan_category_raw        100        0.78820       0.78843     0.00023

saved -> /Users/ruby.grambauer/Documents/DEV/OverbookingAnalyse/reports/tables/00_audit/frequency_folding_experiment.csv


## How to read it

- **Threshold sweep** now starts at `min_count=1` = the genuinely raw categories.
  If AUC is flat across the whole range, the folding cutoff is not load-bearing and
  00's `<100` / `<20` stay justified - now without the earlier "re-folds on top" caveat.
  If LogReg rises at *lower* thresholds, the linear model wants more granularity
  (trees usually shrug).
- **`leak_delta` ~ 0** -> fitting the fold vocabulary on the full frame does not
  meaningfully flatter the hold-out, so 00's full-frame folding is safe to keep.
  If it is consistently positive and non-trivial (> ~0.002), switch the folds to a
  train-fit vocabulary (persist it) like we did for country.
- **Reconstruction caveat (small):** raw values are joined back via the
  `(created, arrival, departure)` triple with §4.2 clipping replicated; 41 duplicate-key
  groups are ambiguous in `ratePlan_name` (2 in `channelCode`) and resolve to the first
  raw row - well under 0.1% of rows, cannot move an AUC at the third decimal.
